# Statistical Inference with Combine

[Combine](https://arxiv.org/abs/2404.06614) is the tool used in CMS to perform statistical inference and produce the results that we publish in our papers.
Based on RooStats and RooFit, it is adopted by more than 90% of the analyses performed in the past few years.
Despite a steep learning curve, it provides everything needed to publish a CMS analysis.
In this notebook we will use the histograms produced in the previous step to perform statistical inference and make statements on the parameters of the theory we are testing.

As you can see from the [documentation](https://cms-analysis.github.io/HiggsAnalysis-CombinedLimit/latest/), the tool mostly relies on a list of scripts with multiple options that have to be called from the terminal.

In [1]:
from IPython.display import display, IFrame
import os

In [2]:
os.makedirs("combine_plots", exist_ok=True)

In [3]:
%env PATH=opt/conda/bin/:/opt/conda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tmp/HiggsAnalysis/CombinedLimit/build/bin
%env LD_LIBRARY_PATH=/opt/conda/lib/:/tmp/HiggsAnalysis/CombinedLimit/build/lib
%env PYTHONPATH=/tmp/HiggsAnalysis/CombinedLimit/build/lib/python

env: PATH=opt/conda/bin/:/opt/conda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tmp/HiggsAnalysis/CombinedLimit/build/bin
env: LD_LIBRARY_PATH=/opt/conda/lib/:/tmp/HiggsAnalysis/CombinedLimit/build/lib
env: PYTHONPATH=/tmp/HiggsAnalysis/CombinedLimit/build/lib/python


## Write the datacard

Combine uses a specific way to describe the likelihood, the so-called datacard. It can be written by hand (which is convenient only when we deal with rather simple analyses) or automate the process based on the coffea output.
At the moment, each analysis seem to have its own way to write the datacard based on the coffea outputs. An attempt to realize a "once for all" procedure is being made in PocketCoffea.

In [4]:
!cat datacard_by_hand.txt

imax *
jmax *
kmax *
-----------------------------------------------------------------------------------------------------------------------------------------------------------------
shapes * * all_histograms_fps4_$CHANNEL.root $PROCESS $PROCESS_$SYSTEMATIC
shapes ttbar * all_histograms_fps4_$CHANNEL.root ttbar ttbar_$SYSTEMATIC
-----------------------------------------------------------------------------------------------------------------------------------------------------------------
bin          bin4j1b      bin4j2b
observation  -1           -1
-----------------------------------------------------------------------------------------------------------------------------------------------------------------
bin      bin4j1b  bin4j1b              bin4j1b              bin4j1b          bin4j1b  bin4j2b  bin4j2b              bin4j2b              bin4j2b          bin4j2b
process  ttbar    single_top_s_chan    single_top_t_chan    single_top_tW    wjets    ttbar    single_top_s_chan    sing

The first part
```
imax *
jmax *
kmax *

```
refers to the number of channels, backgrounds and nuisances parameters respectively. It is possible to add `*` and let the tool figure out on its own, but it fails if this part is not there...

The lines in this part
```
shapes * * all_histograms_fps4_$CHANNEL.root $PROCESS $PROCESS_$SYSTEMATIC
shapes ttbar * all_histograms_fps4_$CHANNEL.root ttbar ttbar_$SYSTEMATIC
```
follow the following format
```
shapes process channel file histogram [histogram_with_systematics]
```
and point to the histograms that we produced in the previous part.

The part
```
bin          bin4j1b      bin4j2b
observation  -1           -1
```
defines the observed rate per channel. One can either write it explicitly or make Combine figure out on its own by integrating (like in this case, by adding `-1`).

The part
```
bin      bin4j1b  bin4j1b              bin4j1b              bin4j1b          bin4j1b  bin4j2b  bin4j2b              bin4j2b              bin4j2b          bin4j2b
process  ttbar    single_top_s_chan    single_top_t_chan    single_top_tW    wjets    ttbar    single_top_s_chan    single_top_t_chan    single_top_tW    wjets
process  0        1                    2                    3                4        0        1                    2                    3                4
rate     -1       -1                   -1                   -1               -1       -1       -1                   -1                   -1               -1
```
defines what is signal and what is backgound, essentially by assigning numbers `<-0` to signal processes and `>0` to backgrounds, and the expected rate (also in this case, a value of `-1` tells Combine to figure out by integrating.

The last section refers to systematic uncertainties. 

The block
```
pt_scale   shape 1 1 1 1 1 1 1 1 1 1
pt_res     shape 1 1 1 1 1 1 1 1 1 1
btag_var_0 shape 1 1 1 1 1 1 1 1 1 1
btag_var_1 shape 1 1 1 1 1 1 1 1 1 1
btag_var_2 shape 1 1 1 1 1 1 1 1 1 1
btag_var_3 shape 1 1 1 1 1 1 1 1 1 1
ME_var     shape 1 0 0 0 0 1 0 0 0 0
PS_var     shape 1 0 0 0 0 1 0 0 0 0
scale      shape 1 0 0 0 0 1 0 0 0 0
scale_var  shape 0 0 0 0 1 0 0 0 0 1
```
is for shape uncertainties (notice that they correspond to the $\pm 1 \sigma$ histograms that we generated during the coffea part.

The block
```
lumi       lnN   1.03 1.03 1.03 1.03 1.03 1.03 1.03 1.03 1.03 1.03
```
describes the type of distribution (log-normal, `lnN`) used to described the nuisance parameter associated to luminosity uncertainty. The following `#channels*#processes` define the relative effect on the rate of each process in each channel (in this case a 3% uncertainty).

The last block
```
bin4j1b autoMCStats 0
bin4j2b autoMCStats 0
```
refers to the statistical uncertainties coming from the amount of events in MC samples. It is described at [this link](https://cms-analysis.github.io/HiggsAnalysis-CombinedLimit/latest/part2/bin-wise-stats/).

## Create workspace

Combine uses a specific script called `text2workspace.py` to create the workspace used for the fit. 
The part
```
--PO 'map=.*/ttbar:r[1.0,0.0,3.0]'
```
scales the signal contribution with a signal strength modifier `r` (usually called $\mu$ in the literature).

In [12]:
!text2workspace.py datacard_by_hand.txt --PO 'map=.*/ttbar:r[1.0,0.0,3.0]'

Channel bin4j1b will use autoMCStats with settings: event-threshold=0, include-signal=0, hist-mode=1
Analyzing bin errors for: prop_binbin4j1b
Poisson cut-off: 0
Processes excluded for sums: ttbar
Bin        Contents        Error           Notes                         
0          94.302246       16.567768       total sum                     
0          35.012794       15.696147       excluding marked processes    
0          5.000000        2.236068        Unweighted events, alpha=18.860449
  => Total parameter prop_binbin4j1b_bin0[0.00,-7.00,7.00] to be Gaussian constrained
------------------------------------------------------------
1          1853.055336     78.037221       total sum                     
1          664.420448      74.337204       excluding marked processes    
1          80.000000       8.944272        Unweighted events, alpha=23.163192
  => Total parameter prop_binbin4j1b_bin1[0.00,-7.00,7.00] to be Gaussian constrained
--------------------------------------------

## Impacts and pulls

In most (all) the analyses approval procedures, it is required to check the impact of the nuisance parameters (NP) on the parameter of interest ($\mu$).
The impact of a NP is defined as the shift $ \Delta \mu$ induced as the NP is fixed to its $\pm 1 \sigma$ values, with all the other parameters profiled as normal.

Another quantity worth checking is the pull, defined as $pull(\theta) = \frac{\hat{\theta} - \theta_0}{\sigma_0}$, which quantifies how far from its expected value we had to "pull" $\theta$ while finding the MLE.

In [13]:
# needed because of https://github.com/cms-analysis/HiggsAnalysis-CombinedLimit/issues/1049
os.environ["CMSSW_BASE"] = "."
os.environ["SCRAM_ARCH"] = "."

The procedure follows these steps:
1. do an initial fit of the POIs with the `--robustFit 1` option

In [14]:
!combineTool.py -M Impacts -d datacard_by_hand.root --robustFit 1 --doInitialFit -m 125

Have POIs: ['r']
>> combine -M MultiDimFit -n _initialFit_Test --algo singles --redefineSignalPOIs r --robustFit 1 -m 125 -d datacard_by_hand.root
 <<< Combine >>> 
 <<< v10.0.0 >>>
>>> Random number generator seed is 123456
>>> Method used is MultiDimFit
Doing initial fit: 

 --- MultiDimFit ---
best fit parameter values and profile-likelihood uncertainties: 
   r :    +0.977   -0.082/+0.090 (68%)
Done in 0.00 min (cpu), 0.00 min (real)


2. perform a scan for each NP

In [15]:
!combineTool.py -M Impacts -d datacard_by_hand.root --robustFit 1 --doFits -m 125

Have POIs: ['r']
Have parameters: 33
>> combine -M MultiDimFit -n _paramFit_Test_ME_var --algo impact --redefineSignalPOIs r -P ME_var --floatOtherPOIs 1 --saveInactivePOI 1 --robustFit 1 -m 125 -d datacard_by_hand.root
 <<< Combine >>> 
 <<< v10.0.0 >>>
>>> Random number generator seed is 123456
>>> Method used is MultiDimFit
Doing initial fit: 

 --- MultiDimFit ---
Parameter impacts: 
  Parameter :   Best-fit               r            
  ME_var    :   +0.906  -0.228/+0.226  -0.038/+0.036
Done in 0.00 min (cpu), 0.00 min (real)
>> combine -M MultiDimFit -n _paramFit_Test_PS_var --algo impact --redefineSignalPOIs r -P PS_var --floatOtherPOIs 1 --saveInactivePOI 1 --robustFit 1 -m 125 -d datacard_by_hand.root
 <<< Combine >>> 
 <<< v10.0.0 >>>
>>> Random number generator seed is 123456
>>> Method used is MultiDimFit
Doing initial fit: 

 --- MultiDimFit ---
Parameter impacts: 
  Parameter :   Best-fit               r            
  PS_var    :   +0.799  -0.255/+0.245  +0.050/-0.051
Don

3. collect and save the output to a JSON file

In [16]:
!combineTool.py -M Impacts -d datacard_by_hand.root --robustFit 1 --output impacts.json -m 125

Have POIs: ['r']
Have parameters: 33


4. make a plot

In [17]:
!plotImpacts.py -i impacts.json -o combine_plots/impacts

>> Doing page 0, have 30 parameters
Info in <TCanvas::Print>: pdf file ./combine_plots/impacts.pdf has been created using the current canvas
TCanvas::Constructor:0: RuntimeWarning: Deleting canvas with same name: combine_plots/impacts
>> Doing page 1, have 3 parameters
Info in <TCanvas::Print>: Current canvas added to pdf file ./combine_plots/impacts.pdf and file closed


In [18]:
pdf_path = "combine_plots/impacts.pdf"
display(IFrame(pdf_path, width=800, height=600))

## Pre- and post-fit distributions

One of the modes Combine can be run with is `FitDiagnostics`. It performs a background-only and a signal+background fit and produces information useful for disgnostics.
One thing that one can do consists in visualizing the distributions we are fitting both before and after the fit (pre- and post- fit).

In [19]:
!combine -M FitDiagnostics datacard_by_hand.root --saveShapes --saveWithUncertainties -n FitDiagnosticsStuff

 <<< Combine >>> 
 <<< v10.0.0 >>>
>>> Random number generator seed is 123456
>>> Method used is FitDiagnostics
[WARNING]: Unable to determine uncertainties on all fit parameters in s+b fit. The option --saveWithUncertainties will be ignored as it would lead to incorrect results. Have a look at https://cms-analysis.github.io/HiggsAnalysis-CombinedLimit/part3/nonstandard/#fit-parameter-uncertainties for more information.

 --- FitDiagnostics ---
Best fit r: 0.976756  -0.0824591/+0.0903623  (68% CL)
Done in 0.05 min (cpu), 0.05 min (real)


After running the fits, we can plot the distributions with one of the scripts taken from the Combine tutorial and lcoated inside `combine_scripts`.

In [20]:
!for region in bin4j1b bin4j2b; do for shape in shapes_prefit shapes_fit_b shapes_fit_s; do python3 combine_scripts/postFitPlot_new.py --input_file fitDiagnosticsFitDiagnosticsStuff.root --shape_type $shape --region $region; done; done

Info in <TCanvas::Print>: png file combine_plots/stacked_plot_shapes_prefit_bin4j1b.png has been created
Info in <TCanvas::Print>: pdf file combine_plots/stacked_plot_shapes_prefit_bin4j1b.pdf has been created
Info in <TCanvas::Print>: png file combine_plots/stacked_plot_shapes_fit_b_bin4j1b.png has been created
Info in <TCanvas::Print>: pdf file combine_plots/stacked_plot_shapes_fit_b_bin4j1b.pdf has been created
Info in <TCanvas::Print>: png file combine_plots/stacked_plot_shapes_fit_s_bin4j1b.png has been created
Info in <TCanvas::Print>: pdf file combine_plots/stacked_plot_shapes_fit_s_bin4j1b.pdf has been created
Info in <TCanvas::Print>: png file combine_plots/stacked_plot_shapes_prefit_bin4j2b.png has been created
Info in <TCanvas::Print>: pdf file combine_plots/stacked_plot_shapes_prefit_bin4j2b.pdf has been created
Info in <TCanvas::Print>: png file combine_plots/stacked_plot_shapes_fit_b_bin4j2b.png has been created
Info in <TCanvas::Print>: pdf file combine_plots/stacked_plo

In [21]:
display(IFrame("combine_plots/stacked_plot_shapes_prefit_bin4j1b.png", width=800, height=600))
display(IFrame("combine_plots/stacked_plot_shapes_fit_s_bin4j1b.png", width=800, height=600))
display(IFrame("combine_plots/stacked_plot_shapes_prefit_bin4j2b.png", width=800, height=600))
display(IFrame("combine_plots/stacked_plot_shapes_fit_s_bin4j2b.png", width=800, height=600))

## Likelihood scan for $\mu$

Another step commonly performed consists in performing an explicit scan of the profile likelihood to provide an interval at the $\alpha$CL around the best fit.
In asymptotic approximation, this done by identifying the parameter values at which the test statistic $q = -2\Delta lnL$ equals a critical value. This value is the $\alpha$ quantile of the $\chi^2$ distribution with one degree of freedom.

The first part consists in performing a global fit whose result we save in a workspace.

In [22]:
!combine -M MultiDimFit datacard_by_hand.root -n .datacard_by_hand.snapshot --rMin -1 --rMax 4 --saveWorkspace

 <<< Combine >>> 
 <<< v10.0.0 >>>
>>> Random number generator seed is 123456
>>> Method used is MultiDimFit
Doing initial fit: 

 --- MultiDimFit ---
best fit parameter values: 
   r :    +0.977
Done in 0.00 min (cpu), 0.00 min (real)


Using the previous result as a starting point, we then perform a scan in a range of the POI.

In [23]:
!combine -M MultiDimFit higgsCombine.datacard_by_hand.snapshot.MultiDimFit.mH120.root -n .datacard_by_hand --rMin 0 --rMax 2 --algo grid --points 80 --snapshotName MultiDimFit

 <<< Combine >>> 
 <<< v10.0.0 >>>
>>> Random number generator seed is 123456
>>> Method used is MultiDimFit
Doing initial fit: 
 POI: r= 0.976722 -> [0,2]
Point 0/80 r = 0.0125
Point 1/80 r = 0.0375
Point 2/80 r = 0.0625
Point 3/80 r = 0.0875
RooAbsMinimizerFcn: Minimized function has error status.
Returning maximum FCN so far (1.30751e+08) to force MIGRAD to back out of this region. Error log follows.
Parameter values: 	ME_var=43.2932	PS_var=-4201.18	btag_var_0=1.11324	btag_var_1=2.6881	btag_var_2=3.79535	btag_var_3=8.77683	lumi=45.9316	pt_res=-31.1457	pt_scale=7.28358	scale=1.34226	scale_var=11.7318

Point 4/80 r = 0.1125
Point 5/80 r = 0.1375
Point 6/80 r = 0.1625
Point 7/80 r = 0.1875
Point 8/80 r = 0.2125
Point 9/80 r = 0.2375
FASTEXIT from pdf_binbin4j1b_obsOnly
FASTEXIT from pdf_binbin4j2b_obsOnly
RooAbsMinimizerFcn: Minimized function has error status.
Returning maximum FCN so far (2.47588e+27) to force MIGRAD to back out of this region. Error log follows.
Parameter values: 	M

In order to discriminate the uncertainty contribution between systematic and statistical, we perform another scan but this time we "freeze" the nuisance parameters to their best fit value.

In [24]:
!combine -M MultiDimFit higgsCombine.datacard_by_hand.snapshot.MultiDimFit.mH120.root -n .datacard_by_hand.freezeAll --rMin 0 --rMax 2 --algo grid --points 800 --snapshotName MultiDimFit --freezeParameters allConstrainedNuisances

 <<< Combine >>> 
 <<< v10.0.0 >>>
>>> Random number generator seed is 123456
>>> Method used is MultiDimFit
Doing initial fit: 
 POI: r= 0.976693 -> [0,2]
Point 0/800 r = 0.00125
Point 1/800 r = 0.00375
Point 2/800 r = 0.00625
Point 3/800 r = 0.00875
Point 4/800 r = 0.01125
Point 5/800 r = 0.01375
Point 6/800 r = 0.01625
Point 7/800 r = 0.01875
Point 8/800 r = 0.02125
Point 9/800 r = 0.02375
Point 10/800 r = 0.02625
Point 11/800 r = 0.02875
Point 12/800 r = 0.03125
Point 13/800 r = 0.03375
Point 14/800 r = 0.03625
Point 15/800 r = 0.03875
Point 16/800 r = 0.04125
Point 17/800 r = 0.04375
Point 18/800 r = 0.04625
Point 19/800 r = 0.04875
Point 20/800 r = 0.05125
Point 21/800 r = 0.05375
Point 22/800 r = 0.05625
Point 23/800 r = 0.05875
Point 24/800 r = 0.06125
Point 25/800 r = 0.06375
Point 26/800 r = 0.06625
Point 27/800 r = 0.06875
Point 28/800 r = 0.07125
Point 29/800 r = 0.07375
Point 30/800 r = 0.07625
Point 31/800 r = 0.07875
Point 32/800 r = 0.08125
Point 33/800 r = 0.08375
Poin

We can then use the scripts provided in the Combine tutorial to get the final results.

In [25]:
!python3 combine_scripts/plot1DScan.py higgsCombine.datacard_by_hand.MultiDimFit.mH120.root --others 'higgsCombine.datacard_by_hand.freezeAll.MultiDimFit.mH120.root:FreezeAll:2' -o combine_plots/likelihood_scan --breakdown Syst,Stat

--------------------------------------
combine_plots/likelihood_scan
--------------------------------------
[{'lo': 0.8943009603167758, 'hi': 1.0671094139359145, 'valid_lo': True, 'valid_hi': True}]
[{'lo': 0.8130792904792637, 'hi': 1.166788089812344, 'valid_lo': True, 'valid_hi': True}]
[{'lo': 0.9727369283122372, 'hi': 0.9806592825863404, 'valid_lo': True, 'valid_hi': True}]
[{'lo': 0.9687898255088606, 'hi': 0.9846345500132655, 'valid_lo': True, 'valid_hi': True}]
Info in <TCanvas::Print>: pdf file ./combine_plots/likelihood_scan.pdf has been created
Info in <TCanvas::Print>: png file ./combine_plots/likelihood_scan.png has been created


In [26]:
display(IFrame("combine_plots/likelihood_scan.png", width=800, height=600))